# Multi-Source Visual Search: MSSearch V2 Benchmark (Google Colab)
### Model: **InternVL3-8B** (`OpenGVLab/InternVL3-8B`)
### Evaluation Pool: **N_1** ($N=1$)

Evaluates the complete **MSSearch V2** pipeline on **STASHBench**.
Integrates SAM3 semantic grounding, SigLIP cross-attention, continuous enclosure sub-canvases, and clean foveated reading.
Results persist incrementally into Google Drive with timestamped filenames and auto-resume.

In [ ]:
# Cell 1: Mount Google Drive & Configure Paths
import os
import sys
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Smart Path Auto-Detection across common Drive locations
POSSIBLE_DATASET_DIRS = [
    '/content/drive/MyDrive/Datasets/STASHBench',
    '/content/drive/MyDrive/STASHBench',
    '/content/drive/MyDrive/cvpr_research/data/STASHBench',
]

DATASET_PATH = None
IMAGE_DIR = None

for d in POSSIBLE_DATASET_DIRS:
    for anno in ['annotations_part1234_merged.json']:
        p = os.path.join(d, anno)
        if os.path.isfile(p):
            DATASET_PATH = p
            break
    if DATASET_PATH:
        for img_sub in ['pool', 'images', '.']:
            ip = os.path.join(d, img_sub)
            if os.path.isdir(ip) and len(os.listdir(ip)) > 0:
                IMAGE_DIR = ip
                break
        break

# Default fallbacks if not auto-detected: edit these paths manually if needed
if not DATASET_PATH:
    DATASET_PATH = '/content/drive/MyDrive/Datasets/STASHBench/annotations_part1234_merged.json'
if not IMAGE_DIR:
    IMAGE_DIR = '/content/drive/MyDrive/Datasets/STASHBench/pool'

MODELS_DIR = '/content/drive/MyDrive/Models'
OUTPUT_DIR = '/content/drive/MyDrive/Mssearch_results/our'

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Dataset Path : {DATASET_PATH} (exists: {os.path.isfile(DATASET_PATH)})')
print(f'Image Dir    : {IMAGE_DIR} (exists: {os.path.isdir(IMAGE_DIR)})')
print(f'Output Dir   : {OUTPUT_DIR}')

In [ ]:
# Cell 2: GPU Sanity Check & Clone/Update Repository
!nvidia-smi

REPO_DIR = '/content/MSSearch'
BRANCH = 'draft/improve_all'

GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') or userdata.get('GH_TOKEN')
    hf_tok = userdata.get('HF_TOKEN')
    if hf_tok:
        os.environ['HF_TOKEN'] = hf_tok
        print('Loaded HF_TOKEN from Colab Secrets.')
    if GITHUB_TOKEN:
        print('Loaded GITHUB_TOKEN from Colab Secrets.')
except Exception:
    pass

if not GITHUB_TOKEN:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')

if GITHUB_TOKEN:
    REPO_URL = f'https://{GITHUB_TOKEN}@github.com/tothanhdat2006/MSSearch.git'
else:
    REPO_URL = 'https://github.com/tothanhdat2006/MSSearch.git'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git remote set-url origin {REPO_URL} && git checkout {BRANCH} && git pull origin {BRANCH}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
    sys.path.insert(0, os.path.join(REPO_DIR, 'experiments', 'our'))

In [ ]:
# Cell 3: Install Dependencies & Download Auxiliary Weights
!pip install --prefer-binary --upgrade-strategy only-if-needed -r {REPO_DIR}/requirements.txt
!python -m spacy download en_core_web_sm --no-deps

# Verify & Auto-download SAM3 if missing
SAM3_DIR = os.path.join(MODELS_DIR, 'sam3')
SAM3_PATH = os.path.join(SAM3_DIR, 'sam3.pt')

if not os.path.isfile(SAM3_PATH):
    print('sam3.pt not found. Downloading from ModelScope...')
    os.makedirs(SAM3_DIR, exist_ok=True)
    tar_tmp = '/content/sam3.tar.gz'
    import urllib.request, tarfile, shutil
    urllib.request.urlretrieve(
        'https://www.modelscope.cn/models/llp1995/sam3/resolve/master/sam3.tar.gz',
        tar_tmp
    )
    print('Extracting sam3.tar.gz...')
    with tarfile.open(tar_tmp, 'r:*') as tar:
        tar.extractall(SAM3_DIR)
    if os.path.exists(tar_tmp):
        os.remove(tar_tmp)
    nested = os.path.join(SAM3_DIR, 'sam3', 'sam3.pt')
    if os.path.isfile(nested) and not os.path.isfile(SAM3_PATH):
        shutil.copyfile(nested, SAM3_PATH)

print(f'SAM3 checkpoint ready at: {SAM3_PATH} (exists: {os.path.isfile(SAM3_PATH)})')

SIGLIP_NAME = os.path.join(MODELS_DIR, 'siglip2-so400m-patch16-512')
if not os.path.isdir(SIGLIP_NAME):
    SIGLIP_NAME = 'google/siglip2-so400m-patch16-512'

NLP_MODEL = 'en_core_web_sm'

In [ ]:
# Cell 4: Launch Evaluation
MODEL_NAME = 'OpenGVLab/InternVL3-8B'  # Target VLM: InternVL3-8B
POOL_MODE = 'N_1'  # Target Pool: N_1 (N=1)
CONFIG_FILE = f'{REPO_DIR}/experiments/our/config_default.json'

# Set index bounds (or None for full benchmark)
START_IDX = None
END_IDX = None
MAX_SAMPLES = None
RESUME = False

cmd = [
    sys.executable, f'{REPO_DIR}/experiments/our/eval_runner.py',
    '--model_path', f'"{MODEL_NAME}"',
    '--sam3_path', f'"{SAM3_PATH}"',
    '--siglip_name', f'"{SIGLIP_NAME}"',
    '--nlp_model', f'"{NLP_MODEL}"',
    '--dataset_path', f'"{DATASET_PATH}"',
    '--image_dir', f'"{IMAGE_DIR}"',
    '--output_dir', f'"{OUTPUT_DIR}"',
    '--config', f'"{CONFIG_FILE}"',
    '--pool_mode', f'"{POOL_MODE}"',
]

if RESUME: cmd.append('--resume')
if START_IDX is not None: cmd.extend(['--start_idx', str(START_IDX)])
if END_IDX is not None: cmd.extend(['--end_idx', str(END_IDX)])
if MAX_SAMPLES is not None: cmd.extend(['--max_samples', str(MAX_SAMPLES)])

full_cmd = ' '.join(cmd)
print(f'Executing: {full_cmd}')
!{full_cmd}